In [51]:
import sys
import os
import re
from pathlib import Path
sys.path.append(os.path.dirname(os.getcwd()))


AUDIO_TIMESTAMP_RE = re.compile(r'^.+\.\d{8}\.\d{4}(\d{2})?$')  # .YYYYMMDD.HHMM or .YYYYMMDD.HHMMSS

In [52]:
from file_management.comprehensive_metadata_and_filesearch import get_file_locations_for_birds
from tools.dbquery import getUUIDfromBands
from tools.system_utils import check_sys_for_macaw_root

In [53]:
def parse_macaw_dirs_file(txt_file_path):
    """Parse MacawAllDirsByBird.txt to get bird -> directories mapping"""
    bird_directories = {}
    
    try:
        with open(txt_file_path, 'r') as file:
            for line in file:
                line = line.strip()
                if line:
                    parts = line.split(',')
                    if len(parts) >= 4:
                        bird_name = parts[0].strip()
                        dir_string = parts[3].strip('[]')
                        directories = [d.strip() for d in dir_string.split('//') if d.strip()]
                        bird_directories[bird_name] = directories
    except FileNotFoundError:
        print(f"File {txt_file_path} not found!")
        return {}
    
    return bird_directories

def get_screening_directories(bird_name, root_dir):
    screening_paths = [
        'public/screening',
        'public/adult_screening',
        'public/from_egret/egret/screening',
        'public/from_stork/stork/screening'
    ]
    
    dirs = []
    for path in screening_paths:
        full = os.path.join(root_dir, path)
        if os.path.exists(full):
            try:
                for sub in os.listdir(full):
                    if bird_name.lower() in sub.lower():
                        dirs.append(os.path.join(full, sub))
            except Exception:
                continue
    return dirs

def is_audio_candidate(filename: str) -> bool:
    name = Path(filename).name.lower()

    if name.endswith(('.wav', '.cbin')):
        return True

    # extensionless recording names like bk100bk99.20070925.0900
    return bool(AUDIO_TIMESTAMP_RE.match(name))


def get_top_songs(birds, txt_path, db_path, root_dir, max_files=5, min_size_mb=0.1):
    """
    For each bird:
    - get file paths from txt via mapping
    - convert to directories
    - scan those directories for plausible audio files
    - pick the largest audio files
    """

    def get_audio_files_from_dirs(dirs):
        files = []
        for d in dirs:
            if not os.path.isdir(d):
                continue
            try:
                for f in os.listdir(d):
                    if not is_audio_candidate(f):
                        continue

                    fp = os.path.join(d, f)
                    if os.path.isfile(fp):
                        try:
                            size = os.path.getsize(fp)
                            if size >= min_size_mb * 1024 * 1024:
                                files.append((fp, size))
                        except OSError:
                            continue
            except Exception:
                continue
        return files

    primary_map = get_file_locations_for_birds(
        birds,
        txt_file_path=txt_path,
        db_path=db_path
    )

    results = {}

    for bird in birds:
        print(f"\nProcessing {bird}...")

        dirs = set()

        for fp in primary_map.get(bird, []):
            fp = fp.replace("\\", "/")
            full_path = fp if os.path.isabs(fp) else os.path.join(root_dir, fp)

            if os.path.exists(full_path):
                dirs.add(os.path.dirname(full_path))

        dirs = list(dirs)[:5]
        print(f"  → searching {len(dirs)} directories")

        files = get_audio_files_from_dirs(dirs)

        files.sort(key=lambda x: x[1], reverse=True)
        top = files[:max_files]

        results[bird] = [
            {
                "filename": os.path.basename(fp),
                "filepath": fp,
                "size_mb": f"{size / (1024 * 1024):.2f}"
            }
            for fp, size in top
        ]

        print(f"  → {len(top)} files selected")

    return results

In [54]:
import pandas as pd

def get_birds_from_nest_summary(csv_path, top_n=3):
    df = pd.read_csv(csv_path)
    df = df.sort_values(by='# XF', ascending=False).head(top_n)
    
    
    birds = set()
    
    for _, row in df.iterrows():
        # --- offspring ---
        if pd.notna(row.get('HR Birds')) and row['HR Birds']:
            birds.update([b.strip() for b in row['HR Birds'].split(';') if b.strip()])
        
        if pd.notna(row.get('XF Birds')) and row['XF Birds']:
            birds.update([b.strip() for b in row['XF Birds'].split(';') if b.strip()])
        
        # --- ADD THESE ---
        if pd.notna(row.get('Nest Father')) and row['Nest Father']:
            birds.add(row['Nest Father'].strip())
        
        if pd.notna(row.get('Genetic Fathers of XF')) and row['Genetic Fathers of XF']:
            birds.update([b.strip() for b in row['Genetic Fathers of XF'].split(';') if b.strip()])

    return sorted(birds)

csv_path = '/Users/annietaylor/Documents/ucsf/brainard/x-foster/nest_father_offspring_summary.csv'
priority_birds = get_birds_from_nest_summary(csv_path)

print(f"{len(priority_birds)} birds found")
print(priority_birds[:10])  # preview

100 birds found
['bk100bk99', 'bk100wh100', 'bk100wh89', 'bk11bk12', 'bk15bk14', 'bk15bk19', 'bk17bk30', 'bk17wh56', 'bk18', 'bk18bk40']


In [56]:
root_dir = check_sys_for_macaw_root()
song_results = get_top_songs(
    birds=priority_birds,
    txt_path="../refs/MacawAllDirsByBird.txt",
    db_path="../refs/2026-01-15-db.sqlite3",
    root_dir=root_dir,
    max_files=5,
    min_size_mb=0.1
)


Processing bk100bk99...
  → searching 5 directories
  → 5 files selected

Processing bk100wh100...
  → searching 1 directories
  → 5 files selected

Processing bk100wh89...
  → searching 5 directories
  → 5 files selected

Processing bk11bk12...
  → searching 1 directories
  → 5 files selected

Processing bk15bk14...
  → searching 0 directories
  → 0 files selected

Processing bk15bk19...
  → searching 0 directories
  → 0 files selected

Processing bk17bk30...
  → searching 2 directories
  → 5 files selected

Processing bk17wh56...
  → searching 3 directories
  → 5 files selected

Processing bk18...
  → searching 0 directories
  → 0 files selected

Processing bk18bk40...
  → searching 0 directories
  → 0 files selected

Processing bk1bk3...
  → searching 5 directories
  → 5 files selected

Processing bk20bk45...
  → searching 5 directories
  → 5 files selected

Processing bk23wh94...
  → searching 0 directories
  → 0 files selected

Processing bk24wh25...
  → searching 3 directories
 

In [57]:
import json
import os

def save_song_results(song_results, filepath):
    """
    Save song_results dictionary to JSON
    """
    try:
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(song_results, f, indent=2)
        print(f"Saved song_results to: {filepath}")
    except Exception as e:
        print(f"Error saving file: {e}")

save_song_results(song_results, 'priority_bird_songpaths.json')

Saved song_results to: priority_bird_songpaths.json


In [ ]:
def check_birds_in_txt(bird_list, txt_path):
    txt_birds = set()
    
    with open(txt_path, 'r') as f:
        for line in f:
            line = line.strip()
            if line:
                parts = line.split(',')
                if len(parts) >= 1:
                    txt_birds.add(parts[0].strip())
    
    found = []
    missing = []
    
    for bird in bird_list:
        if bird in txt_birds:
            found.append(bird)
        else:
            missing.append(bird)
    
    print(f"Total birds: {len(bird_list)}")
    print(f"Found in txt: {len(found)}")
    print(f"Missing from txt: {len(missing)}")
    
    print("\nExample FOUND:")
    print(found[:10])
    
    print("\nExample MISSING:")
    print(missing[:10])
    
    return found, missing

In [47]:
found, missing = check_birds_in_txt(
    bird_list,
    "../refs/MacawAllDirsByBird.txt"
)

Total birds: 112
Found in txt: 86
Missing from txt: 26

Example FOUND:
['bk100bk99', 'bk100wh100', 'bk100wh89', 'bk11bk12', 'bk13bk12', 'bk17bk30', 'bk17wh56', 'bk18bk40', 'bk1bk3', 'bk20bk45']

Example MISSING:
['bk15bk14', 'bk15bk19', 'bk18', 'bk2wh3', 'bk38bk2', 'bk41bk11', 'bk4wh78', 'bk65wh20', 'bk66wh37', 'bk67bk95']


In [24]:
from file_management.comprehensive_metadata_and_filesearch import generate_bird_name_variations
txt_path = "../refs/MacawAllDirsByBird.txt"

txt_name_to_dirs = {}
txt_variation_map = {}

with open(txt_path, 'r') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        parts = line.split(',')
        if len(parts) < 4:
            continue

        txt_name = parts[0].strip()
        dir_string = parts[3].strip('[]')
        dirs = [d.strip() for d in dir_string.split('//') if d.strip()]

        txt_name_to_dirs[txt_name] = dirs

        # generate variations for THIS txt name
        variations = generate_bird_name_variations(txt_name)

        for var in variations:
            txt_variation_map[var] = txt_name

In [25]:
matched = {}
still_missing = []

for bird in missing:
    bird_vars = generate_bird_name_variations(bird)

    found_match = None

    for var in bird_vars:
        if var in txt_variation_map:
            found_match = txt_variation_map[var]
            break

    if found_match:
        matched[bird] = found_match
    else:
        still_missing.append(bird)
        

In [27]:
print(f"Recovered via naming variations: {len(matched)}")
print(f"Still missing: {len(still_missing)}")

print("\nExamples recovered:")
for k, v in list(matched.items())[:10]:
    print(f"{k} → {v}")

print("\nStill missing examples:")
print(still_missing)


Recovered via naming variations: 0
Still missing: 96

Examples recovered:

Still missing examples:
['bk15bk14', 'bk15bk19', 'bk18', 'bk23wh13', 'bk2wh3', 'bk33wh45', 'bk38bk2', 'bk41bk11', 'bk4wh78', 'bk65wh20', 'bk66wh37', 'bk67bk95', 'bk69wh31', 'bk70bk62', 'bk74wh76', 'bk79bk76', 'bk81wh20', 'bk91wh31', 'bk93bk97', 'bk97bk86', 'gr61', 'pk1bk31', 'pk37bk19', 'pk41bk66', 'pk48bk68', 'pk7bk27', 'pk86bk94', 'pk91bk68', 'pu22wh17', 'pu40pu75', 'pu61wh52', 'pu62wh35', 'pu68wh32', 'pu84wh4', 'rd15gr82', 'rd18gr37', 'rd25wh83', 'rd28gr49', 'rd30gr38', 'rd30gr5', 'rd35wh39', 'rd36gr39', 'rd37wh40', 'rd38gr36', 'rd39gr23', 'rd44wh13', 'rd50gr1', 'rd57gr34', 'rd58gr48', 'rd58wh18', 'rd59wh41', 'rd5gr49', 'rd65pu23', 'rd68wh16', 'rd70wh17', 'rd75wh75', 'rd80wh90', 'rd8wh70', 'wh19or81', 'wh24pk92', 'wh2pk4', 'wh34pk92', 'wh37or29', 'wh40or14', 'wh47or16', 'wh47wh87', 'wh48pk16', 'wh48wh88', 'wh49wh89', 'wh50wh90', 'wh51or57', 'wh58or62', 'wh5or8', 'wh64or100', 'wh64or55', 'wh65or78', 'wh66or61'

In [19]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile

from tools.evfuncs import load_cbin


def save_spectrograms(song_results, output_root="xfoster_specs"):
    base_dir = os.path.join(os.getcwd(), output_root)
    os.makedirs(base_dir, exist_ok=True)

    for bird, files in song_results.items():
        bird_dir = os.path.join(base_dir, bird)
        os.makedirs(bird_dir, exist_ok=True)

        print(f"\nProcessing spectrograms for {bird}...")

        for file_info in files:
            filepath = file_info["filepath"]
            filename = file_info["filename"]

            base_name = os.path.basename(filepath)
            out_path = os.path.join(bird_dir, f"{base_name}.png")

            try:
                # --- LOAD AUDIO ---
                if filepath.endswith(".cbin"):
                    audio, sr = load_cbin(filepath)

                else:
                    try:
                        sr, audio = wavfile.read(filepath)
                        if len(audio.shape) > 1:
                            audio = audio[:, 0]
                    except Exception:
                        # fallback to librosa if needed
                        import librosa
                        audio, sr = librosa.load(filepath, sr=None)

                # --- MAKE SPECTROGRAM ---
                plt.figure(figsize=(6, 4))
                plt.specgram(audio, Fs=sr, NFFT=1024, noverlap=512)
                plt.axis('off')

                # Optional label
                plt.title(f"{bird}", fontsize=8)

                # --- SAVE ---
                plt.savefig(out_path, bbox_inches='tight', pad_inches=0)
                plt.close()

            except Exception as e:
                print(f"  Failed: {filepath}")
                print(f"    Error: {e}")

In [58]:
#save_spectrograms(song_results)

In [29]:
cohort_df = pd.read_pickle("cohort_df.pkl")

In [30]:
birth_lookup = {
    row['Bird Name']: pd.to_datetime(row['Birth Date'])
    for _, row in cohort_df.iterrows()
}

rearing_lookup = {
    row['Bird Name']: row['Rearing Type']
    for _, row in cohort_df.iterrows()
}

nest_lookup = {
    row['Bird Name']: row['Nest Father']
    for _, row in cohort_df.iterrows()
}

gen_lookup = {
    row['Bird Name']: row['Genetic Father']
    for _, row in cohort_df.iterrows()
}

In [ ]:
records = []

for bird, files in song_results.items():
    for i, f in enumerate(files, 1):
        filepath = f["filepath"]
        filename = f["filename"]

        # spectrogram path
        spec_path = os.path.join(
            os.getcwd(),
            "xfoster_specs",
            bird,
            f"{filename}.png"
        )

        # age
        rec_date = parse_recording_date(filepath)
        birth_date = birth_lookup.get(bird)

        age_days = None
        if rec_date and birth_date:
            age_days = (rec_date - birth_date).days

        # role (simple version for now)
        role = "offspring"

        records.append({
            "id": f"{bird}_{i:02d}",   # useful for blinding
            "bird": bird,
            "filename": filename,
            "filepath": filepath,
            "spectrogram_path": spec_path,
            "rearing_type": rearing_lookup.get(bird, "unknown"),
            "nest_father": nest_lookup.get(bird),
            "genetic_father": gen_lookup.get(bird),
            "role": role,
            "age_days": age_days
        })

In [ ]:
import random
random.shuffle(records)

In [ ]:
for rec in records:
    if not os.path.exists(rec["spectrogram_path"]):
        print("Missing:", rec["spectrogram_path"])

In [ ]:
ages = [r["age_days"] for r in records if r["age_days"] is not None]
print(min(ages), max(ages))

In [ ]:
!pip install streamlit

In [ ]:
import streamlit as st
import pandas as pd
import random

# Load records (from your earlier step)
records = pd.read_pickle("records.pkl")

# Shuffle once
if "order" not in st.session_state:
    st.session_state.order = records.sample(frac=1).reset_index(drop=True)
    st.session_state.idx = 0
    st.session_state.responses = []

rec = st.session_state.order.iloc[st.session_state.idx]

st.image(rec["spectrogram_path"], use_container_width=True)

st.write(f"ID: {rec['id']}")

col1, col2, col3 = st.columns(3)

if col1.button("Category A"):
    st.session_state.responses.append((rec["id"], "A"))

if col2.button("Category B"):
    st.session_state.responses.append((rec["id"], "B"))

if col3.button("Skip"):
    st.session_state.responses.append((rec["id"], "skip"))

if st.button("Next"):
    st.session_state.idx += 1